# SetFit: Regressão Logística vs. SVM calibrado

Compara duas cabeças sobre o mesmo encoder BERTimbau treinado por SetFit: a Regressão Logística padrão do SetFit e `CalibratedClassifierCV(LinearSVC)`. Em cada fold externo, o encoder contrastivo é treinado uma vez; as duas cabeças recebem os mesmos embeddings do fold de treino e são avaliadas nos mesmos emails de teste.

A calibração sigmoid usa validação cruzada estratificada de 3 folds somente dentro do treino externo (`ensemble=False`). O fold externo nunca participa no ajuste do encoder, no treino do SVM ou na calibração. Com poucas dezenas de emails, os resultados são exploratórios: a calibração e as métricas podem variar bastante entre amostras.

In [1]:
import gc
import json
import logging
import os
import sys
import tempfile
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from packaging.version import Version
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

try:
    from datasets import Dataset
    from setfit import SetFitModel, Trainer as SetFitTrainer
    from setfit import TrainingArguments as SetFitTrainingArguments
except ImportError as error:
    raise RuntimeError('Instala as dependências com: pip install -r requirements.txt') from error

if Version(version('setfit')) < Version('1.2'):
    raise RuntimeError('Este notebook requer setfit>=1.2.')
if not torch.cuda.is_available():
    raise RuntimeError('Este experimento SetFit requer uma GPU NVIDIA com CUDA.')

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Executa o notebook na raiz do projeto ou dentro de research/.')
sys.path.insert(0, str(ROOT))

from transformers import set_seed
from src.email_data import LABELS
from src.finetuning import (
    DEFAULT_EMAILS_DIR, DEFAULT_EXCEL, clear_device_cache,
    dataset_fingerprint, load_examples, stratified_folds,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
for logger_name in ('setfit', 'sentence_transformers', 'transformers'):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

DEVICE = 'cuda'
BASE_MODEL = 'neuralmind/bert-base-portuguese-cased'
SEED = 42
N_FOLDS = 5
SETFIT_EPOCHS = 3
SETFIT_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
BODY_LEARNING_RATE = 2e-5
MAX_LENGTH = 512
CALIBRATION_FOLDS = 3
BOOTSTRAP_RESAMPLES = 5000

/home/alex/Desktop/classification_globalbrico/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
examples = load_examples(DEFAULT_EXCEL, [DEFAULT_EMAILS_DIR])
folds = stratified_folds(examples, N_FOLDS, SEED)
fold_by_uid = {
    uid: fold_index
    for fold_index, fold in enumerate(folds, start=1)
    for uid, _text, _label in fold
}
assert len(fold_by_uid) == len(examples)

print(f'Emails: {len(examples)}; folds externos: {N_FOLDS}')
print('Distribuição de labels:', pd.Series([label for _uid, _text, label in examples]).value_counts().to_dict())

Emails: 41; folds externos: 5
Distribuição de labels: {'SPAM': 16, 'Pedido de Informação': 14, 'Pedido de Encomenda': 11}


In [3]:
def make_dataset(rows):
    return Dataset.from_dict({
        'text': [text for _uid, text, _label in rows],
        'label': [LABELS.index(label) for _uid, _text, label in rows],
    })


def run_head_comparison():
    prediction_rows = []
    fold_rows = []

    for fold_index, test_examples in enumerate(folds, start=1):
        test_uids = {uid for uid, _text, _label in test_examples}
        train_examples = [row for row in examples if row[0] not in test_uids]
        fold_seed = SEED + fold_index - 1
        train_texts = [text for _uid, text, _label in train_examples]
        test_texts = [text for _uid, text, _label in test_examples]
        y_train = np.asarray([LABELS.index(label) for _uid, _text, label in train_examples])
        y_test = np.asarray([LABELS.index(label) for _uid, _text, label in test_examples])
        class_counts = np.bincount(y_train, minlength=len(LABELS))
        if class_counts.min() < CALIBRATION_FOLDS:
            raise ValueError(f'Fold {fold_index}: exemplos insuficientes por classe para calibração: {class_counts}.')

        set_seed(fold_seed)
        train_dataset = make_dataset(train_examples)
        print(f'Fold {fold_index}/{N_FOLDS}: treino={len(train_examples)}, teste={len(test_examples)}')
        with tempfile.TemporaryDirectory(prefix=f'setfit-head-fold-{fold_index}-') as temporary:
            model = SetFitModel.from_pretrained(BASE_MODEL, labels=list(LABELS), device=DEVICE)
            model.model_body.max_seq_length = MAX_LENGTH
            training_args = SetFitTrainingArguments(
                output_dir=temporary,
                batch_size=SETFIT_BATCH_SIZE,
                num_epochs=SETFIT_EPOCHS,
                body_learning_rate=BODY_LEARNING_RATE,
                max_length=MAX_LENGTH,
                sampling_strategy='oversampling',
                use_amp=True,
                show_progress_bar=False,
                logging_strategy='no',
                save_strategy='no',
                report_to='none',
                seed=fold_seed,
            )
            trainer = SetFitTrainer(model=model, args=training_args, train_dataset=train_dataset)
            trainer.train()

            if not isinstance(model.model_head, LogisticRegression):
                raise TypeError(f'Cabeça SetFit esperada LogisticRegression, recebida {type(model.model_head).__name__}.')
            train_embeddings = np.asarray(model.encode(train_texts, batch_size=EVAL_BATCH_SIZE, show_progress_bar=False))
            test_embeddings = np.asarray(model.encode(test_texts, batch_size=EVAL_BATCH_SIZE, show_progress_bar=False))

            logistic_probabilities = model.model_head.predict_proba(test_embeddings)
            logistic_class_ids = model.model_head.classes_[np.argmax(logistic_probabilities, axis=1)]
            calibration_cv = StratifiedKFold(
                n_splits=CALIBRATION_FOLDS, shuffle=True, random_state=fold_seed,
            )
            calibrated_svm = CalibratedClassifierCV(
                estimator=LinearSVC(C=1.0, class_weight='balanced', random_state=fold_seed),
                method='sigmoid', cv=calibration_cv, ensemble=False,
            )
            calibrated_svm.fit(train_embeddings, y_train)
            svm_probabilities = calibrated_svm.predict_proba(test_embeddings)
            svm_class_ids = calibrated_svm.classes_[np.argmax(svm_probabilities, axis=1)]

            for index, ((uid, _text, real_label), logistic_id, logistic_probability, svm_id, svm_probability) in enumerate(zip(
                test_examples, logistic_class_ids, logistic_probabilities, svm_class_ids, svm_probabilities,
            )):
                prediction_rows.extend([
                    {
                        'model': 'SetFit + Regressão Logística', 'uid': uid, 'fold': fold_index,
                        'label_real': real_label, 'label_prevista': LABELS[int(logistic_id)],
                        'score': float(logistic_probability[np.argmax(logistic_probabilities[index])]),
                    },
                    {
                        'model': 'SetFit + SVM linear calibrado', 'uid': uid, 'fold': fold_index,
                        'label_real': real_label, 'label_prevista': LABELS[int(svm_id)],
                        'score': float(svm_probability[np.argmax(svm_probabilities[index])]),
                    },
                ])

            fold_rows.append({
                'fold': fold_index, 'n_train': len(train_examples), 'n_test': len(test_examples),
                'calibration_cv_folds': CALIBRATION_FOLDS, 'seed': fold_seed,
            })
            del trainer, model, train_dataset, train_embeddings, test_embeddings
            del calibrated_svm, logistic_probabilities, svm_probabilities
            clear_device_cache()
            gc.collect()

    predictions = pd.DataFrame(prediction_rows).sort_values(['model', 'uid']).reset_index(drop=True)
    expected_uids = {uid for uid, _text, _label in examples}
    for model_name, group in predictions.groupby('model'):
        if len(group) != len(examples) or set(group['uid']) != expected_uids or group['uid'].nunique() != len(examples):
            raise RuntimeError(f'{model_name}: previsões não cobrem cada UID exatamente uma vez.')
        if dict(zip(group['uid'], group['fold'].astype(int))) != fold_by_uid:
            raise RuntimeError(f'{model_name}: folds das previsões não correspondem aos folds externos.')
    return predictions, pd.DataFrame(fold_rows)


predictions, fold_details = run_head_comparison()

Fold 1/5: treino=32, teste=9


Map: 100%|██████████| 32/32 [00:00<00:00, 7046.29 examples/s]


{'embedding_loss': '0.05868', 'grad_norm': '1.058', 'learning_rate': '0', 'epoch': '0.002976'}
{'train_runtime': '288.5', 'train_samples_per_second': '6.987', 'train_steps_per_second': '3.493', 'train_loss': '0.05413', 'epoch': '3'}
Fold 2/5: treino=33, teste=8


Map: 100%|██████████| 33/33 [00:00<00:00, 17355.74 examples/s]


{'embedding_loss': '0.3246', 'grad_norm': '1.866', 'learning_rate': '0', 'epoch': '0.002786'}
{'train_runtime': '295.3', 'train_samples_per_second': '7.294', 'train_steps_per_second': '3.647', 'train_loss': '0.04475', 'epoch': '3'}
Fold 3/5: treino=33, teste=8


...

{'embedding_loss': '0.3584', 'grad_norm': '1.702', 'learning_rate': '0', 'epoch': '0.002786'}
{'train_runtime': '145.3', 'train_samples_per_second': '14.83', 'train_steps_per_second': '7.414', 'train_loss': '0.04385', 'epoch': '3'}
Fold 4/5: treino=33, teste=8


Map: 100%|██████████| 33/33 [00:00<00:00, 18308.47 examples/s]


{'embedding_loss': '0.3745', 'grad_norm': '1.728', 'learning_rate': '0', 'epoch': '0.002786'}
{'train_runtime': '146.9', 'train_samples_per_second': '14.67', 'train_steps_per_second': '7.333', 'train_loss': '0.03947', 'epoch': '3'}
Fold 5/5: treino=33, teste=8


Map: 100%|██████████| 33/33 [00:00<00:00, 9385.78 examples/s]


{'embedding_loss': '0.6387', 'grad_norm': '2.771', 'learning_rate': '0', 'epoch': '0.002786'}
{'train_runtime': '144.6', 'train_samples_per_second': '14.9', 'train_steps_per_second': '7.449', 'train_loss': '0.04702', 'epoch': '3'}


In [4]:
def summarize_predictions(frame):
    rows = []
    for model_name, group in frame.groupby('model', sort=False):
        truth = group['label_real']
        predicted = group['label_prevista']
        _, recalls, _, _ = precision_recall_fscore_support(
            truth, predicted, labels=LABELS, zero_division=0,
        )
        row = {
            'Modelo': model_name,
            'Accuracy': accuracy_score(truth, predicted),
            'Macro-F1': f1_score(truth, predicted, labels=LABELS, average='macro', zero_division=0),
        }
        row.update({f'Recall — {label}': recalls[index] for index, label in enumerate(LABELS)})
        rows.append(row)
    return pd.DataFrame(rows).sort_values('Macro-F1', ascending=False).reset_index(drop=True)


summary = summarize_predictions(predictions)
display(summary.style.format({column: '{:.3f}' for column in summary.columns if column != 'Modelo'}))


def paired_bootstrap(n_resamples=BOOTSTRAP_RESAMPLES):
    logistic = predictions[predictions['model'] == 'SetFit + Regressão Logística']
    svm = predictions[predictions['model'] == 'SetFit + SVM linear calibrado']
    paired = logistic.merge(svm, on='uid', suffixes=('_logistic', '_svm'), validate='one_to_one')
    if not (paired['label_real_logistic'] == paired['label_real_svm']).all():
        raise AssertionError('As labels reais diferem entre cabeças.')
    truth = paired['label_real_logistic'].to_numpy()
    logistic_pred = paired['label_prevista_logistic'].to_numpy()
    svm_pred = paired['label_prevista_svm'].to_numpy()
    rng = np.random.default_rng(SEED)
    deltas = np.empty((n_resamples, 2))
    for index in range(n_resamples):
        sample = rng.integers(0, len(paired), len(paired))
        deltas[index, 0] = accuracy_score(truth[sample], svm_pred[sample]) - accuracy_score(truth[sample], logistic_pred[sample])
        deltas[index, 1] = f1_score(truth[sample], svm_pred[sample], labels=LABELS, average='macro', zero_division=0) - f1_score(truth[sample], logistic_pred[sample], labels=LABELS, average='macro', zero_division=0)
    return pd.DataFrame([{
        'Comparação': 'SVM calibrado - Regressão Logística',
        'Delta accuracy': deltas[:, 0].mean(),
        'IC95% delta accuracy': tuple(np.quantile(deltas[:, 0], [0.025, 0.975]).round(3)),
        'Delta macro-F1': deltas[:, 1].mean(),
        'IC95% delta macro-F1': tuple(np.quantile(deltas[:, 1], [0.025, 0.975]).round(3)),
    }])


uncertainty = paired_bootstrap()
display(uncertainty)

,Modelo,Accuracy,Macro-F1,Recall — Pedido de Informação,Recall — Pedido de Encomenda,Recall — SPAM
0,SetFit + Regressão Logística,0.585,0.573,0.714,0.455,0.562
1,SetFit + SVM linear calibrado,0.585,0.573,0.714,0.455,0.562


,Comparação,Delta accuracy,IC95% delta accuracy,Delta macro-F1,IC95% delta macro-F1
0,SVM calibrado - Regressão Logística,0.0,"(0.0, 0.0)",0.0,"(0.0, 0.0)"


In [5]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = ROOT / 'data/setfit_head_comparison' / run_id
suffix = 1
while output_dir.exists():
    output_dir = ROOT / 'data/setfit_head_comparison' / f'{run_id}-{suffix}'
    suffix += 1
output_dir.mkdir(parents=True)

predictions.to_csv(output_dir / 'predictions.csv', index=False, encoding='utf-8-sig')
summary.to_csv(output_dir / 'summary.csv', index=False, encoding='utf-8-sig')
uncertainty.to_csv(output_dir / 'paired_bootstrap.csv', index=False, encoding='utf-8-sig')
fold_details.to_csv(output_dir / 'folds.csv', index=False, encoding='utf-8-sig')
configuration = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_fingerprint': dataset_fingerprint(examples),
    'n_examples': len(examples), 'labels': list(LABELS),
    'seed': SEED, 'outer_folds': N_FOLDS,
    'setfit': {
        'base_model': BASE_MODEL, 'epochs': SETFIT_EPOCHS,
        'batch_size': SETFIT_BATCH_SIZE, 'body_learning_rate': BODY_LEARNING_RATE,
        'max_length': MAX_LENGTH, 'device': DEVICE,
    },
    'heads': {
        'reference': 'SetFit default LogisticRegression',
        'candidate': 'CalibratedClassifierCV(LinearSVC)',
        'calibration_method': 'sigmoid', 'calibration_folds': CALIBRATION_FOLDS,
        'calibration_ensemble': False, 'svm_C': 1.0, 'svm_class_weight': 'balanced',
    },
    'folds_by_uid': fold_by_uid,
}
(output_dir / 'configuration.json').write_text(
    json.dumps(configuration, ensure_ascii=False, indent=2), encoding='utf-8',
)
print(f'Resultados guardados em: {output_dir}')

Resultados guardados em: /home/alex/Desktop/classification_globalbrico/data/setfit_head_comparison/20260928T105419Z
